Lấy 3 chỉ số chứng khoán, so sánh mức độ tăng giảm so với ngày hôm trước

In [3]:
from vnstock import Quote
import pandas as pd
from datetime import timedelta,time

symbols = ['VNINDEX', 'VN30', 'HNX30']
data_frames = []

for sym in symbols:
    quote = Quote(symbol=sym, source='VCI')
    df = quote.history(length='1', interval='d').reset_index()  # reset index để có cột time thay vì index
    # chèn cột mới ở vị trí 0 (đầu DataFrame)
    df.insert(0, 'symbol', sym)
    # df = df.df[['symbol', 'time', 'close', 'volume']]
    # trong df, lấy ra các cột cần thiết: symbol, time, close, volume
    df = df[['symbol', 'time', 'close', 'volume']]
    data_frames.append(df)
    
all_data = pd.concat(data_frames, ignore_index=True)

# trong từng symsbol, lấy giá của ngày d-1, gán vào ngày d-2, tính chênh lệch điểm và phần trăm thay đổi
all_data['prev_close'] = all_data.groupby('symbol')['close'].shift(1)
all_data['change'] = all_data['close'] - all_data['prev_close']
all_data['change_pct'] = (all_data['change'] / all_data['prev_close'] * 100).round(2)	

today = pd.Timestamp.now().date().strftime('%Y-%m-%d')
all_data_today = all_data[all_data['time'] == today]

all_data_today.reset_index()
# all_data.to_csv('index_data.csv', index=False)

,index,symbol,time,close,volume,prev_close,change,change_pct
0,2,VNINDEX,2026-03-16,1693.21,873491825,1696.24,-3.03,-0.18
1,5,VN30,2026-03-16,1852.99,346778683,1853.60,-0.61,-0.03
2,8,HNX30,2026-03-16,529.66,69242727,524.78,4.88,0.93


In [7]:
all_data_today.to_csv('index_data.csv', index=False)

# 2. lấy top 10 cổ phiếu vốn hóa giảm

In [ ]:
# code lấy top 10 cổ phiếu có vốn hóa giảm/tăng
from datetime import datetime, timedelta
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
vn30 = pd.read_csv('ticker_vn30.csv')           
vn30 = vn30['symbol'].astype(str) + ".VN"  
# thêm ".VN" vào mỗi mã để phù hợp với yfinance

# check ngày làm việc
# nếu thứ 7, chủ nhật, thứ 2: lấy giá thứ 5& 6
# nếu thứ 3: lấy giá thứ 6 và thứ 2 
def get_date(d):
    if d.weekday() == 0:
        d1= d - timedelta(days=4)
        d2= d - timedelta(days=2)
    elif 1 <= d.weekday <= 5:
        d1= d - timedelta(days=2)
        d2= d - timedelta(days=0)
    elif d.weekday() == 6:
        d1= d - timedelta(days=3)
        d2= d - timedelta(days=1)
    return d1,d2

get_date(datetime.today())
d = get_date(datetime.today())
start = d[0].strftime("%Y-%m-%d")
end = d[1].strftime("%Y-%m-%d")

# start=(datetime.today() - timedelta(days=3)).strftime("%Y-%m-%d")
# end= datetime.today().strftime("%Y-%m-%d")
# trong yfinance,lấy giá đóng cửa của ngày start và end
# từ đó tính ra thay đổi điểm và phần trăm thay đổi của cổ phiếu trong khoảng thời gian đó

top_10_high = []
for ticker in vn30:
	data = yf.download(ticker, start=start, end=end)
	sharesOutstanding = yf.Ticker(ticker).info.get('sharesOutstanding', 'N/A')
	if data.empty:
		print(f"No data for {ticker} between {start} and {end}.")
		continue
	close_start = data['Close'].iloc[0]
	close_end = data['Close'].iloc[-1]
	sharesOutstanding = sharesOutstanding
	change_points = close_end - close_start
	change_percent = (change_points / close_start)*100
	market_cap_d_2ays_ago = (close_start * sharesOutstanding)/1e9
	maket_cap_now = (close_end * sharesOutstanding)/1e9
	change_market_cap_bil_vnd = maket_cap_now - market_cap_d_2ays_ago
	top_10_high.append({
		'Ticker': ticker.replace(".VN", "")
		,'Close End': close_end
		,'Change Points': change_points
		,'Change Percent': change_percent
		,'Market Cap 2 Days Ago (Bil VND)': market_cap_d_2ays_ago
		,'Market Cap Now (Bil VND)': maket_cap_now
		,'Change Market Cap (Bil VND)': change_market_cap_bil_vnd
		})

top_10_high= pd.DataFrame(top_10_high)
# Chuyển các cột về kiểu float để chỉ hiển thị số
top_10_high['Close End'] = top_10_high['Close End'].astype(float)
top_10_high['Change Points'] = top_10_high['Change Points'].astype(float)
top_10_high['Change Percent'] = top_10_high['Change Percent'].astype(float)
top_10_high['Market Cap 2 Days Ago (Bil VND)'] = top_10_high['Market Cap 2 Days Ago (Bil VND)'].astype(float)
top_10_high['Market Cap Now (Bil VND)'] = top_10_high['Market Cap Now (Bil VND)'].astype(float)
top_10_high['Change Market Cap (Bil VND)'] = top_10_high['Change Market Cap (Bil VND)'].astype(float)

top_10_high.sort_values(by='Change Market Cap (Bil VND)', ascending=False, inplace=True)
top_10_high.reset_index(drop=True, inplace=True)


In [88]:
top_10_high.head()

,Ticker,Close End,Change Points,Change Percent,Market Cap 2 Days Ago (Bil VND),Market Cap Now (Bil VND),Change Market Cap (Bil VND)
0,VHM,98000.0,2100.0,2.189781,393900.811184,402526.376392,8625.565208
1,VNM,63100.0,1500.0,2.435065,128741.255412,131876.188580,3134.933168
2,VJC,156800.0,4000.0,2.617801,90398.211835,92764.657171,2366.445336
3,ACB,23450.0,300.0,1.295896,118913.600267,120454.597247,1540.996980
4,LPB,41500.0,400.0,0.973236,122777.294310,123972.207150,1194.912840


In [89]:
top_10_high.to_csv('top_10_high.csv')

# 3. get stock high performance

In [10]:
df = pd.read_csv('ticker_hose.csv')
df = df['symbol'].astype(str) + ".VN"

In [ ]:
scanner = []
for ticker in df:
    info = yf.Ticker(ticker).info
    price = info.get('regularMarketPrice', '')
    ROE = info.get('returnOnEquity', '')*100
    ROA = info.get('returnOnAssets', '')*100
    ROIC = info.get('returnOnInvestedCapital', '')*100
    PE = info.get('trailingPE', '')
    PB = info.get('priceToBook','')
    EPS = info.get('trailingEps','')
    
    scanner.append({
    'Ticker': ticker.replace(".VN", "")
    ,'Price': price
    ,'ROE': ROE
    ,'ROA': ROA
    ,'ROIC':ROIC
    ,'PE': PE
    ,'PB': PB
    ,'EPS': EPS

  })
scanner_df = pd.DataFrame(scanner)

for col in ['ROE','ROA','ROIC','PE','PB','EPS']:
    scanner_df[col] = pd.to_numeric(scanner_df[col], errors='coerce')


In [ ]:
# Tạo cột xếp hạng roe, pe, pb
scanner_df['ROE_Rank'] = scanner_df['ROE'].rank(method='dense',ascending=False, na_option='bottom')
scanner_df['PE_Rank'] = scanner_df['PE'].rank(method='dense',ascending=True, na_option='bottom')
scanner_df['ROE + PE'] = (scanner_df['ROE'].rank(method='dense',ascending=False, na_option='bottom') 
                          + 
                          (scanner_df['PE'].rank(method='dense',ascending=True, na_option='bottom')))
scanner_df['ROE + PE Ranking'] = scanner_df['ROE + PE'].rank(method='dense',ascending=True, na_option='bottom').astype(int)
scanner_df.sort_values(by='ROE + PE Ranking',ascending=True).reset_index()

In [54]:
scanner_df.to_csv('ranking_ticker.csv')

# 4. Lấy 3 bảng: income_statement, balance_sheet, valuation_measures

In [ ]:
def get_income_statement(ticker):
    """Get income statement for a single ticker, return as a flat DataFrame with Ticker and Date columns."""
    stmt = yf.Ticker(ticker).get_income_stmt(freq='quarterly')
    df = pd.DataFrame(stmt).loc[['TotalRevenue', 'CostOfRevenue', 'GrossProfit', 'NetIncomeContinuousOperations']]
    
    records = []
    for date_col in df.columns:
        row = {
            'Ticker': ticker.replace(".VN",""),
            'Date': date_col.strftime('%Y-%m-%d'),
            'TotalRevenue': df.at['TotalRevenue', date_col] / 1e9,
            'CostOfRevenue': df.at['CostOfRevenue', date_col] / 1e9,
            'GrossProfit': df.at['GrossProfit', date_col] / 1e9,
            'NetIncome': df.at['NetIncomeContinuousOperations', date_col] / 1e9,
        }
        row['GrossProfitMargin'] = 100.0*round(row['GrossProfit'] / row['TotalRevenue'], 2) if row['TotalRevenue'] else None
        row['NetIncomeMargin'] = 100.0*round(row['NetIncome'] / row['TotalRevenue'], 2) if row['TotalRevenue'] else None
        records.append(row)
    
    result = pd.DataFrame(records).sort_values('Date').reset_index(drop=True)
    return result

def get_income_statements(tickers):
    """Get income statements for multiple tickers and merge into one DataFrame."""
    all_data = []
    for ticker in tickers:
        try:
            df = get_income_statement(ticker)
            all_data.append(df)
            print(f"OK: {ticker}")
        except Exception as e:
            print(f"SKIP: {ticker} - {e}")
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

tickers = df
merged_stmt = get_income_statements(tickers)
merged_stmt

In [91]:
# 1 số stock không share dữ liệu lên yfinance
merged_stmt = merged_stmt.dropna(subset=['TotalRevenue','CostOfRevenue','GrossProfit','NetIncome','GrossProfitMargin','NetIncomeMargin'])
merged_stmt.head()

,Ticker,Date,TotalRevenue,CostOfRevenue,GrossProfit,NetIncome,GrossProfitMargin,NetIncomeMargin
1,AAA,2024-09-30,3193.399535,2848.063507,345.336028,-25.701898,11.0,-1.0
2,AAA,2024-12-31,3842.519613,3393.914631,448.604982,63.675468,12.0,2.0
3,AAA,2025-03-31,3856.408102,3392.693587,463.714514,55.512421,12.0,1.0
4,AAA,2025-06-30,2309.974268,1955.834761,354.139506,171.766008,15.0,7.0
5,AAA,2025-12-31,2192.291126,1864.920072,327.371053,62.021847,15.0,3.0


In [57]:
merged_stmt.to_csv('income_statement.csv')

In [ ]:
def get_balance_sheet(ticker):
    """Get income statement for a single ticker, return as a flat DataFrame with Ticker and Date columns."""
    bls = yf.Ticker(ticker).get_balance_sheet(freq='quarterly')
    df = pd.DataFrame(bls).loc[['TotalAssets','TotalLiabilitiesNetMinorityInterest','TotalEquityGrossMinorityInterest']]
    
    records = []
    for date_col in df.columns:
        row = {
            'Ticker': ticker.replace(".VN",""),
            'Date': date_col.strftime('%Y-%m-%d'),
            'Total Assets': df.at['TotalAssets', date_col] / 1e9,
            'Total Liabilities': df.at['TotalLiabilitiesNetMinorityInterest', date_col] / 1e9,
            'Total Equity': df.at['TotalEquityGrossMinorityInterest', date_col] / 1e9,
        }
        row['Liabilities/Equity ratio'] = round(row['Total Liabilities'] / row['Total Equity'], 2) if row['Total Equity'] else None

        records.append(row)
    
    result = pd.DataFrame(records).sort_values('Date').reset_index(drop=True)
    return result

def get_balance_sheets(tickers):
    """Get income statements for multiple tickers and merge into one DataFrame."""
    all_data = []
    for ticker in tickers:
        try:
            df = get_balance_sheet(ticker)
            all_data.append(df)
            print(f"OK: {ticker}")
        except Exception as e:
            print(f"SKIP: {ticker} - {e}")
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

tickers = df
merged_balance_sheet = get_balance_sheets(tickers)
merged_balance_sheet.dropna(subset= ['Total Assets','Total Liabilities','Total Equity','Liabilities/Equity ratio'])

In [59]:
merged_balance_sheet.to_csv('balance_sheet.csv')

# 5. lấy giá và sma20

In [11]:
import yfinance as yf
import pandas as pd
import talib
from datetime import datetime,timedelta


start = ((datetime.today() - timedelta(days=120)).replace(day=1).strftime("%Y-%m-%d"))
end = datetime.today().strftime("%Y-%m-%d")

def get_price(ticker):
    """Get price history for a single ticker, return DataFrame with Ticker and Date columns."""
    hist = yf.download(ticker, start=start, end=end, interval='1d')
    close = hist['Close'].reset_index()
    df = close.melt(var_name='Ticker', id_vars='Date', value_name='Close')
    df = df.sort_values(by=['Ticker', 'Date'])
    df['Ticker'] = df['Ticker'].str.replace('.VN', '')
    df['SMA20'] = talib.SMA(df['Close'], timeperiod=20)
    return df

def get_prices(tickers):
    all_hist = []
    for ticker in tickers:
        get_hist = get_price(ticker)
        all_hist.append(get_hist)
    result = pd.concat(all_hist).sort_values(by=['Ticker','Date']).reset_index(drop=True)
    return result

hist_price = get_prices(df)
hist_price

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,Date,Ticker,Close,SMA20
0,2025-11-03,AAA,8000.0,NaN
1,2025-11-04,AAA,8120.0,NaN
2,2025-11-05,AAA,8070.0,NaN
3,2025-11-06,AAA,8120.0,NaN
4,2025-11-07,AAA,7950.0,NaN
...,...,...,...,...
33570,2026-03-09,YEG,10850.0,12157.5
33571,2026-03-10,YEG,10750.0,12077.5
33572,2026-03-11,YEG,11050.0,12000.0
33573,2026-03-12,YEG,10900.0,11930.0


In [11]:
hist_price.to_csv('hist_price_data.csv')

In [3]:
pip install oauth2client

  Using cached oauth2client-4.1.3-py2.py3-none-any.whl.metadata (1.2 kB)
  Using cached httplib2-0.31.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached rsa-4.9.1-py3-none-any.whl.metadata (5.6 kB)
Using cached oauth2client-4.1.3-py2.py3-none-any.whl (98 kB)
Using cached httplib2-0.31.2-py3-none-any.whl (91 kB)
Using cached rsa-4.9.1-py3-none-any.whl (34 kB)

   ---------------------------------------- 0/3 [rsa]
   ---------------------------------------- 0/3 [rsa]
   ---------------------------------------- 0/3 [rsa]
   ---------------------------------------- 0/3 [rsa]
   ---------------------------------------- 0/3 [rsa]
   ---------------------------------------- 0/3 [rsa]
   ---------------------------------------- 0/3 [rsa]
   ---------------------------------------- 0/3 [rsa]
   -------------------------- ------------- 2/3 [oauth2client]
   -------------------------- ------------- 2/3 [oauth2client]
   -------------------------- ------------- 2/3 [oauth2client]
   -------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
import gspread

In [ ]:
from operator import index

from matplotlib.pyplot import hist
import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials

scope = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive"
]

creds = ServiceAccountCredentials.from_json_keyfile_name(
    "polar-winter-343402-402d8025ab6f.json", scope
)

client = gspread.authorize(creds)
spreadsheet = client.open("project_vnstock_phamduytan")
from operator import index

from matplotlib.pyplot import hist
import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials

scope = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive"
]

creds = ServiceAccountCredentials.from_json_keyfile_name(
    "polar-winter-343402-402d8025ab6f.json", scope
)

client = gspread.authorize(creds)
spreadsheet = client.open("project_vnstock_phamduytan")
spreadsheet = client.open("project_vnstock_phamduytan")

index_sheet = spreadsheet.worksheet("index_data")
hist_sheet = spreadsheet.worksheet("hist_price_data")
# top_10_sheet = spreadsheet.worksheet("top_10_high")
# all_data_today["date"] = all_data_today["date"].dt.strftime("%Y-%m-%d")

# clear dữ liệu cũ
# index_sheet.clear()
hist_sheet.clear()
# top_10_sheet.clear()
#top_10_high
# upload dataframe
# index_sheet.update(
#     "A1",
#     [all_data_today.columns.tolist()] + all_data_today.values.tolist()
# )

hist_sheet.update(
    "A1",hist_price
)

# top_10_sheet.update(
#     "A1",
#     [top_10_high.columns.tolist()] + top_10_high.values.tolist()
# )

print("Data uploaded to Google Sheets successfully!")

In [ ]:
from datetime import datetime, timedelta

def prev_business_day(d):
    d -= timedelta(days=1)
    while d.weekday() >= 5:      # 5 = Sat, 6 = Sun
        d -= timedelta(days=1)
    return d

today = datetime.today()
d1 = prev_business_day(today)    # ngày gần nhất trước today
d2 = prev_business_day(d1)  

start = d1.strftime("%Y-%m-%d") 
end = d2.strftime("%Y-%m-%d") 



'2026-03-12'

In [ ]:
from datetime import date

from requests import get


def get_dates(d):
    today = datetime.today()
    weekday = today.weekday()  # Monday=0, Sunday=6
    while weekday >=5:
        if 1 <= weekday <= 4:  # Thứ 3 (1) đến Thứ 6 (4)
            start = today - timedelta(days=1)
            end = today - timedelta(days=2)
        elif weekday == 0:  # Thứ 2
            start = today - timedelta(days=3)
            end = today - timedelta(days=4)

    return start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")

d = datetime.today()
get_dates(d)



In [ ]:
from datetime import date,timedelta
from turtle import st

def get_date(d):
    if d.weekday() == 0:
        d1= d - timedelta(days=4)
        d2= d - timedelta(days=3)
    elif 1 <= d.weekday <= 5:
        d1= d - timedelta(days=2)
        d2= d - timedelta(days=1)
    elif d.weekday() == 6:
        d1= d - timedelta(days=3)
        d2= d - timedelta(days=2)
    return d1,d2

get_date(datetime.today())
# get_date(datetime.today()-timedelta(days=1))
# # type(d1)
# get_date(cn)
start = d1[1].strftime("%Y-%m-%d")

end = d1[0].strftime("%Y-%m-%d")

# # # print start và end
# print("start:",start)
# print("end:",end)

'2026-03-12'